# ShortCutDesign Theoretical Stages Validation Test

Test notebook to verify the issue of negative theoretical stages (N_stages) in the ShortCutDesign model.

In [1]:
import sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Import ShortCutDesign class
sys.path.append('/Users/k23070952/MultiFidelity-ProcessOpt/Process/1. Code')
from ShortCutDesign import ShortCutDesign

## Define Test Cases

Design variable X is an 8-dimensional vector:
- X[0]: n_stages (extractor stages, 10-50, integer)
- X[1]: Lr1 (extract distiller light key recovery, 0-0.9999)
- X[2]: Hr1 (extract distiller heavy key recovery, 0-0.9999)
- X[3]: Lr2 (acid purification light key recovery, 0-0.9999)
- X[4]: Hr2 (acid purification heavy key recovery, 0-0.9999)
- X[5]: T_hex (heat exchanger temperature, 273-350 K)
- X[6]: Lr3 (raffinate distiller light key recovery, 0-0.9999)
- X[7]: Hr3 (raffinate distiller heavy key recovery, 0-0.9999)

In [2]:
# Define test cases
test_cases = {
    'Base case (default)': [12, 0.95, 0.95, 0.999, 0.999, 310, 0.99, 0.99],
    'Low recovery rates': [15, 0.5, 0.5, 0.7, 0.7, 300, 0.6, 0.6],
    'High recovery rates': [20, 0.99, 0.99, 0.999, 0.999, 320, 0.999, 0.999],
    'Low temperature': [12, 0.95, 0.95, 0.999, 0.999, 280, 0.99, 0.99],
    'High temperature': [12, 0.95, 0.95, 0.999, 0.999, 340, 0.99, 0.99],
    'Min stages': [10, 0.95, 0.95, 0.999, 0.999, 310, 0.99, 0.99],
    'Max stages': [50, 0.95, 0.95, 0.999, 0.999, 310, 0.99, 0.99],
    'Extreme low recoveries': [12, 0.1, 0.1, 0.3, 0.3, 310, 0.2, 0.2],
    'Mixed recoveries': [15, 0.9, 0.5, 0.95, 0.7, 310, 0.8, 0.6],
}

## ShortCutDesign Model Test

In [3]:
def test_shortcut_design(X, case_name):
    """
    Function to test ShortCutDesign model and print results
    """
    print(f"\n{'='*70}")
    print(f"Test Case: {case_name}")
    print(f"{'='*70}")
    print(f"Input X: {X}")
    print()
    
    try:
        # Create ShortCutDesign instance
        model = ShortCutDesign(verbose=False)
        
        # Call shortcut_results() to check theoretical stages
        results = model.shortcut_results(X)
        
        # Print results (handle None values gracefully)
        print("\n[Results]")
        capex_str = f"{results['CAPEX']:.4f}" if results['CAPEX'] is not None else "N/A (Error)"
        opex_str = f"{results['OPEX']:.4f}" if results['OPEX'] is not None else "N/A (Error)"
        purity_str = f"{results['AceticAcidWt']*100:.2f}" if results['AceticAcidWt'] is not None else "N/A (Error)"
        print(f"  CAPEX: {capex_str} MM USD/yr")
        print(f"  OPEX: {opex_str} MM USD/yr")
        print(f"  Acetic Acid Purity: {purity_str} wt%")
        print(f"  Simulation Time: {results['shortcut_time']:.4f} sec")
        print()
        
        print("[Extract Distiller (ED)]")
        print(f"  N_stages_1: {results['N_stages_1']}")
        print(f"  feed_stage_1: {results['feed_stage_1']}")
        boilup_1_str = f"{results['boilup_1']:.4f}" if results['boilup_1'] is not None and not np.isnan(results['boilup_1']) else "N/A"
        print(f"  boilup_1: {boilup_1_str}")
        print()
        
        print("[Acetic Acid Purification (ED2)]")
        print(f"  N_stages_2: {results['N_stages_2']}")
        print(f"  feed_stage_2: {results['feed_stage_2']}")
        boilup_2_str = f"{results['boilup_2']:.4f}" if results['boilup_2'] is not None and not np.isnan(results['boilup_2']) else "N/A"
        print(f"  boilup_2: {boilup_2_str}")
        print()
        
        print("[Raffinate Distiller (RD)]")
        print(f"  N_stages_3: {results['N_stages_3']}")
        print(f"  feed_stage_3: {results['feed_stage_3']}")
        boilup_3_str = f"{results['boilup_3']:.4f}" if results['boilup_3'] is not None and not np.isnan(results['boilup_3']) else "N/A"
        print(f"  boilup_3: {boilup_3_str}")
        print()
        
        # Check for negative stages
        negative_stages = []
        if results['N_stages_1'] < 0:
            negative_stages.append('N_stages_1')
        if results['N_stages_2'] < 0:
            negative_stages.append('N_stages_2')
        if results['N_stages_3'] < 0:
            negative_stages.append('N_stages_3')
        
        if negative_stages:
            print(f"⚠️  Warning: The following stages are negative: {', '.join(negative_stages)}")
        else:
            print("✓ All theoretical stages are positive.")
        
        return results
        
    except Exception as e:
        print(f"\n❌ Error occurred: {e}")
        import traceback
        traceback.print_exc()
        return None

## Run All Test Cases

In [4]:
# List to store results
all_results = []

# Run each test case
for case_name, X in test_cases.items():
    results = test_shortcut_design(X, case_name)
    if results is not None:
        all_results.append({
            'case': case_name,
            **results,
            'X': X
        })


Test Case: Base case (default)
Input X: [12, 0.95, 0.95, 0.999, 0.999, 310, 0.99, 0.99]

 ##### An instance of the 'BlackBox' class  has been initialised!
[12, 0.95, 0.95, 0.999, 0.999, 310, 0.99, 0.99]

[Results]
  CAPEX: 2.1854 MM USD/yr
  OPEX: 0.7712 MM USD/yr
  Acetic Acid Purity: 99.18 wt%
  Simulation Time: 1.0675 sec

[Extract Distiller (ED)]
  N_stages_1: 22
  feed_stage_1: 17
  boilup_1: 2.9662

[Acetic Acid Purification (ED2)]
  N_stages_2: 26
  feed_stage_2: 21
  boilup_2: 9.7106

[Raffinate Distiller (RD)]
  N_stages_3: 8
  feed_stage_3: 1
  boilup_3: 0.0913

✓ All theoretical stages are positive.

Test Case: Low recovery rates
Input X: [15, 0.5, 0.5, 0.7, 0.7, 300, 0.6, 0.6]

 ##### An instance of the 'BlackBox' class  has been initialised!
[15, 0.5, 0.5, 0.7, 0.7, 300, 0.6, 0.6]

[Results]
  CAPEX: 1.4755 MM USD/yr
  OPEX: 1.4892 MM USD/yr
  Acetic Acid Purity: 10.25 wt%
  Simulation Time: 0.0738 sec

[Extract Distiller (ED)]
  N_stages_1: 5
  feed_stage_1: 4
  boilup_1

## Results Summary Table

In [5]:
# Convert to DataFrame and summarize
if all_results:
    df = pd.DataFrame(all_results)
    
    # Select and display key results
    summary_cols = ['case', 'N_stages_1', 'N_stages_2', 'N_stages_3', 
                    'feed_stage_1', 'feed_stage_2', 'feed_stage_3',
                    'AceticAcidWt', 'CAPEX', 'OPEX']
    
    print("\n" + "="*100)
    print("Overall Results Summary")
    print("="*100)
    display(df[summary_cols])
    
    # Filter cases with negative stages
    negative_cases = df[(df['N_stages_1'] < 0) | (df['N_stages_2'] < 0) | (df['N_stages_3'] < 0)]
    
    if not negative_cases.empty:
        print("\n" + "="*100)
        print("⚠️  Cases with Negative Stages")
        print("="*100)
        display(negative_cases[summary_cols])
    else:
        print("\n✓ All test cases produced positive theoretical stages.")


Overall Results Summary


,case,N_stages_1,N_stages_2,N_stages_3,feed_stage_1,feed_stage_2,feed_stage_3,AceticAcidWt,CAPEX,OPEX
0,Base case (default),22,26,8,17,21,1,0.992,2.19,0.771
1,Low recovery rates,5,12,5,4,11,1,0.103,1.48,1.49
2,High recovery rates,32,26,13,26,19,1,0.995,2.74,0.769
3,Low temperature,22,26,9,17,22,1,0.991,2.19,0.891
4,High temperature,22,26,8,18,21,1,0.993,2.15,0.702
5,Min stages,22,26,8,17,21,1,0.992,2.19,0.771
6,Max stages,22,26,8,17,21,1,0.992,2.19,0.771
7,Extreme low recoveries,-8,-2,1,-5,-1,1,0.00252,NaN,5.32
8,Mixed recoveries,11,20,6,7,13,1,0.682,1.99,1.52



⚠️  Cases with Negative Stages


,case,N_stages_1,N_stages_2,N_stages_3,feed_stage_1,feed_stage_2,feed_stage_3,AceticAcidWt,CAPEX,OPEX
7,Extreme low recoveries,-8,-2,1,-5,-1,1,0.00252,NaN,5.32


## Detailed Analysis of Specific Cases

If there are cases with negative stages, analyze the input variables and results in detail.

In [6]:
# Test with custom X values
# You can enter problematic X values here to test.

# Example:
custom_X = [12, 0.95, 0.95, 0.999, 0.999, 310, 0.99, 0.99]
custom_results = test_shortcut_design(custom_X, "Custom Test Case")


Test Case: Custom Test Case
Input X: [12, 0.95, 0.95, 0.999, 0.999, 310, 0.99, 0.99]

 ##### An instance of the 'BlackBox' class  has been initialised!
[12, 0.95, 0.95, 0.999, 0.999, 310, 0.99, 0.99]

[Results]
  CAPEX: 2.1854 MM USD/yr
  OPEX: 0.7712 MM USD/yr
  Acetic Acid Purity: 99.18 wt%
  Simulation Time: 0.0592 sec

[Extract Distiller (ED)]
  N_stages_1: 22
  feed_stage_1: 17
  boilup_1: 2.9662

[Acetic Acid Purification (ED2)]
  N_stages_2: 26
  feed_stage_2: 21
  boilup_2: 9.7106

[Raffinate Distiller (RD)]
  N_stages_3: 8
  feed_stage_3: 1
  boilup_3: 0.0913

✓ All theoretical stages are positive.


## Random Sampling Test

Perform random sampling to test more cases.

In [7]:
from scipy.stats import qmc

# Generate 10 samples using Latin Hypercube Sampling
bounds = [(10, 50), (0, 0.9999), (0, 0.9999), (0, 0.9999), (0, 0.9999), (273, 350), (0, 0.9999), (0, 0.9999)]

n_samples = 10
sampler = qmc.LatinHypercube(d=8)
samples = sampler.random(n=n_samples)

# Scale samples to actual bounds
l_bounds = np.array([b[0] for b in bounds])
u_bounds = np.array([b[1] for b in bounds])
X_samples = qmc.scale(samples, l_bounds, u_bounds)

# Convert first variable to integer
X_samples[:, 0] = np.round(X_samples[:, 0])

print(f"\n{n_samples} random samples generated successfully")
print("\nStarting sample tests...\n")


10 random samples generated successfully

Starting sample tests...



In [8]:
# Random sample test
random_results = []

for i, X in enumerate(X_samples):
    results = test_shortcut_design(X.tolist(), f"Random Sample {i+1}")
    if results is not None:
        random_results.append({
            'sample_id': i+1,
            **results
        })

# Random Sample Results Summary
if random_results:
    df_random = pd.DataFrame(random_results)
    
    print("\n" + "="*100)
    print("Random Sample Results Summary")
    print("="*100)
    display(df_random[['sample_id', 'N_stages_1', 'N_stages_2', 'N_stages_3', 
                        'feed_stage_1', 'feed_stage_2', 'feed_stage_3', 'AceticAcidWt']])
    
    # Negative stages statistics
    n_negative_1 = (df_random['N_stages_1'] < 0).sum()
    n_negative_2 = (df_random['N_stages_2'] < 0).sum()
    n_negative_3 = (df_random['N_stages_3'] < 0).sum()
    
    print(f"\nNegative Stages Statistics:")
    print(f"  N_stages_1 negative: {n_negative_1}/{len(df_random)} ({n_negative_1/len(df_random)*100:.1f}%)")
    print(f"  N_stages_2 negative: {n_negative_2}/{len(df_random)} ({n_negative_2/len(df_random)*100:.1f}%)")
    print(f"  N_stages_3 negative: {n_negative_3}/{len(df_random)} ({n_negative_3/len(df_random)*100:.1f}%)")


Test Case: Random Sample 1
Input X: [33.0, 0.5592512484999519, 0.970145804059361, 0.7149106827177838, 0.263474811695474, 338.9660667003437, 0.5050049608628747, 0.5980312880752892]

 ##### An instance of the 'BlackBox' class  has been initialised!
[33.0, 0.5592512484999519, 0.970145804059361, 0.7149106827177838, 0.263474811695474, 338.9660667003437, 0.5050049608628747, 0.5980312880752892]

[Results]
  CAPEX: 1.3610 MM USD/yr
  OPEX: 1.3299 MM USD/yr
  Acetic Acid Purity: 6.76 wt%
  Simulation Time: 0.0875 sec

[Extract Distiller (ED)]
  N_stages_1: 7
  feed_stage_1: 6
  boilup_1: 0.7647

[Acetic Acid Purification (ED2)]
  N_stages_2: 4
  feed_stage_2: 4
  boilup_2: 2.5510

[Raffinate Distiller (RD)]
  N_stages_3: 5
  feed_stage_3: 1
  boilup_3: 0.6883

✓ All theoretical stages are positive.

Test Case: Random Sample 2
Input X: [25.0, 0.6478983209349912, 0.13997177307009687, 0.967111632321366, 0.6531381659184967, 303.72483717561914, 0.05189427232138242, 0.2073155957348226]

 ##### An in

,sample_id,N_stages_1,N_stages_2,N_stages_3,feed_stage_1,feed_stage_2,feed_stage_3,AceticAcidWt
0,1,7,4,5,6,4,1,0.0676
1,2,1,21,-1,1,18,0,0.334
2,3,-9,1,7,-5,1,1,0.00346
3,4,3,6,5,3,6,1,0.0433
4,5,6,13,3,5,11,1,0.201
5,6,0,0,0,0,0,0,0
6,7,8,-8,7,7,-6,1,0.00707
7,8,9,8,7,6,7,1,0.0988
8,9,16,-2,4,10,-1,1,0.0811
9,10,-6,7,6,-3,7,1,0.0331



Negative Stages Statistics:
  N_stages_1 negative: 2/10 (20.0%)
  N_stages_2 negative: 2/10 (20.0%)
  N_stages_3 negative: 1/10 (10.0%)
